# Embeddings locais das cartas (Etapa 2) — Qwen3-Embedding-0.6B

*(Feito com ClaudeAI)*


- **Modelo:** `Qwen/Qwen3-Embedding-0.6B` (vetores de 1024 dimensões, em vez de 384). Exige `transformers>=4.51.0` e `sentence-transformers>=2.7.0`.
- **Consulta com instrução:** o Qwen3 espera `Instruct: <tarefa>\nQuery:<consulta>` **só do lado da consulta**; os documentos entram sem instrução.
- **Limite de entrada ajustável** (`MAX_SEQ_LENGTH`). O padrão é 1024 e há uma célula que conta quantas cartas ainda seriam truncadas.
- **GPU:** detecta CUDA e usa meia precisão.
- **Blocos com checkpoint:** se o job for interrompido (limite de tempo, falta de memória), ao rodar de novo ele continua de onde parou.
- **Modo offline:** para nós de computação sem internet.
- **Metadados completos:** o arquivo de metadados guarda também o modelo, o prefixo da consulta e uma assinatura dos textos, para o notebook de busca usar exatamente a mesma configuração.

## Rodando no HPC

1. **Ambiente.** Precisa de `torch` (com CUDA), `sentence-transformers>=2.7.0`, `transformers>=4.51.0` e `numpy`. Se você vai rodar como job em lote, também `nbconvert`.
2. **Internet.** Nós de computação costumam não ter internet. Nesse caso:
   - em um nó **com** internet (por exemplo, o de login), rode a célula de download abaixo com `RUN_DOWNLOAD = True`;
   - depois, na configuração, aponte `MODEL_PATH` para a pasta baixada e coloque `OFFLINE = True`.
3. **Cache.** Se a sua HOME tem cota pequena, defina `HF_HOME` para uma pasta em scratch/projeto (linha comentada na configuração).
4. **GPU.** Peça uma GPU no job. Sem GPU, o notebook avisa e roda em CPU, o que é bem mais lento.
5. **Interrupções.** Ao reiniciar depois de um erro ou limite de tempo, basta rodar tudo de novo: os blocos já gravados em `CHECKPOINT_DIR` são pulados. Se faltar memória da GPU, diminua `BATCH_SIZE` e rode outra vez (o `BATCH_SIZE` não invalida os checkpoints).

Exemplo de script de job (**modelo genérico, se o seu HPC usa Slurm** — ajuste partição, conta, módulos e caminhos ao seu cluster):

```bash
#!/bin/bash
#SBATCH --job-name=mtg_embeddings
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
#SBATCH --time=02:00:00

# module load ...        # se o cluster usar módulos
# source activate meu_env
export HF_HOME=/caminho/no/scratch/hf_cache
export HF_HUB_OFFLINE=1   # so se o modelo ja estiver baixado

jupyter nbconvert --to notebook --execute embeddings_busca_mtg_qwen3.ipynb \
    --ExecutePreprocessor.timeout=-1 --output executado_qwen3.ipynb
```

## Configuração

In [1]:
import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np

# ---------------------------------------------------------------- dados
CARD_DOCUMENTS_PATH = "Dados/Embeddings2/card_documents.jsonl"
TEXT_FIELD = "embedding_text"

# ---------------------------------------------------------------- modelo
MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"   # identificador no Hugging Face (vai para o arquivo de metadados)
MODEL_PATH = MODEL_ID                     # ou a pasta do modelo baixado, ex.: "modelos/Qwen3-Embedding-0.6B"
MAX_SEQ_LENGTH = 1024                     # o BGE cortava em 512; ajuste conforme a celula de diagnostico
BATCH_SIZE = 32                           # falta de memoria da GPU? diminua (16, 8...) e rode de novo
CHUNK_SIZE = 2048                         # a cada quantas cartas gravar um checkpoint

# Instrucao da CONSULTA (em ingles, como o cartao do modelo recomenda). Documentos NAO levam instrucao.
QUERY_TASK = ("Given a Magic: The Gathering card search query, "
              "retrieve card descriptions whose rules text matches the query")
QUERY_PREFIX = f"Instruct: {QUERY_TASK}\nQuery:"

# ---------------------------------------------------------------- saidas (nomes proprios: nao sobrescrevem os do BGE)
EMBEDDINGS_PATH = "Dados/embeddings_qwen3.npy"
EMBEDDINGS_META_PATH = "Dados/embeddings_qwen3_meta.json"
CHECKPOINT_DIR = "Dados/checkpoints_qwen3"

# ---------------------------------------------------------------- HPC
# Guardar o cache do Hugging Face fora da HOME (descomente e ajuste):
# os.environ["HF_HOME"] = "/caminho/no/scratch/hf_cache"
OFFLINE = False   # True em no de computacao sem internet (o modelo precisa estar baixado antes)
if OFFLINE:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

## Carregar os documentos de carta

Mesma regra do notebook anterior: cartas com `embedding_text_structured` vazio são puladas, e guardamos `id` e `name` para religar cada vetor à carta.

**Confira o número impresso.** Ele precisa ser o mesmo que o notebook de busca carrega (o `assert` de lá acusa qualquer diferença). Se o `card_documents.jsonl` deste ambiente for de outra versão, os vetores ficarão desalinhados.

In [2]:
ids, names, texts = [], [], []
total_lines = 0
with open(CARD_DOCUMENTS_PATH, encoding="utf-8") as f:
    for line in f:
        total_lines += 1
        card = json.loads(line)
        text = card.get(TEXT_FIELD)
        if not text:
            continue
        names.append(card["name"])
        texts.append(text)

print(f"{len(texts)} cartas carregadas (de {total_lines} no arquivo)")

33147 cartas carregadas (de 33147 no arquivo)


## (Opcional) Baixar o modelo para uso offline

Rode **uma vez, em um nó com internet**, e só se os nós de computação não tiverem internet. Depois disso, ajuste `MODEL_PATH` e `OFFLINE` na configuração.

In [9]:
RUN_DOWNLOAD = False   # troque para True SO em um no COM internet, uma unica vez

if RUN_DOWNLOAD:
    from huggingface_hub import snapshot_download
    local = snapshot_download(MODEL_ID, local_dir="modelos/Qwen3-Embedding-0.6B")
    print("Modelo baixado em:", local)
    print('Agora: MODEL_PATH = "modelos/Qwen3-Embedding-0.6B" e OFFLINE = True (celula de configuracao)')
else:
    print("Download desativado (RUN_DOWNLOAD = False).")

Download desativado (RUN_DOWNLOAD = False).


## Carregar o modelo

Usa GPU se houver (bfloat16 quando suportado, senão float16) e `padding_side="left"`, como recomenda o cartão do modelo. A célula imprime a precisão **realmente** carregada.

In [5]:
import torch
from sentence_transformers import SentenceTransformer

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device, dtype = "cpu", torch.float32
    print("ATENCAO: nenhuma GPU visivel -- a geracao vai ser MUITO mais lenta. Confira se o job pediu GPU.")


def load_model(dtype_key):
    return SentenceTransformer(
        MODEL_PATH,
        device=device,
        model_kwargs={dtype_key: dtype},
        tokenizer_kwargs={"padding_side": "left"},
    )


try:
    model = load_model("dtype")            # transformers recentes
except Exception:
    model = load_model("torch_dtype")      # transformers mais antigos
model.max_seq_length = MAX_SEQ_LENGTH

print(f"Modelo carregado: {MODEL_PATH} | max_seq_length={model.max_seq_length} | "
      f"precisao carregada: {next(model.parameters()).dtype} | device: {device}")

c:\Users\joaquim25018\.conda\envs\Token\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The `tokenizer_kwargs` argument was renamed and is now deprecated. Please use `processor_kwargs` instead.


ATENCAO: nenhuma GPU visivel -- a geracao vai ser MUITO mais lenta. Confira se o job pediu GPU.


c:\Users\joaquim25018\.conda\envs\Token\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\joaquim25018\.cache\huggingface\hub\models--Qwen--Qwen3-Embedding-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 310/310 [00:02<00:00, 117.40it/s]


Modelo carregado: Qwen/Qwen3-Embedding-0.6B | max_seq_length=1024 | precisao carregada: torch.float32 | device: cpu


## Diagnóstico: quantas cartas seriam truncadas?

Conta os tokens de cada documento com o tokenizador do próprio modelo. Se a porcentagem acima de `MAX_SEQ_LENGTH` for alta, aumente o valor na configuração (mais memória e mais tempo por lote).

In [6]:
lens = np.array([len(x) for x in model.tokenizer(texts, truncation=False)["input_ids"]])
over = lens > MAX_SEQ_LENGTH
print(f"tokens por carta -> mediana {np.median(lens):.0f} | p95 {np.percentile(lens, 95):.0f} | maximo {lens.max()}")
print(f"cartas acima de MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}: {over.sum()} ({over.mean():.2%}) -- serao truncadas")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1178 > 1024). Running this sequence through the model will result in indexing errors


tokens por carta -> mediana 96 | p95 625 | maximo 2141
cartas acima de MAX_SEQ_LENGTH=1024: 212 (0.64%) -- serao truncadas


## Teste de tempo (opcional)

Estimativa grosseira do tempo total, a partir de 256 cartas (depois de um aquecimento). Como o comprimento dos textos varia, o valor real pode diferir.

In [7]:
n = min(256, len(texts))
model.encode(texts[:16], batch_size=BATCH_SIZE, normalize_embeddings=True, convert_to_tensor=True)   # aquecimento
t0 = time.time()
model.encode(texts[:n], batch_size=BATCH_SIZE, normalize_embeddings=True, convert_to_tensor=True)
dt = time.time() - t0
print(f"{n} textos em {dt:.1f}s -> estimativa: ~{dt / n * len(texts) / 60:.1f} min para as {len(texts)} cartas")

256 textos em 550.9s -> estimativa: ~1188.9 min para as 33147 cartas


## Gerar os embeddings, em blocos com checkpoint

Cada bloco de `CHUNK_SIZE` cartas é salvo em `CHECKPOINT_DIR` assim que termina. Se o job cair, rodar esta célula de novo pula os blocos já prontos.

Os checkpoints têm uma **assinatura** (modelo, `MAX_SEQ_LENGTH` e todos os textos). Se você mudar qualquer um deles — por exemplo, depois de limpar o texto dos documentos —, os checkpoints antigos são apagados automaticamente, para não misturar vetores de versões diferentes.

Os documentos são embedados **sem instrução**. A instrução vale só para a consulta.

In [ ]:
# GERAR EM BLOCOS (com checkpoint)
ckpt = Path(CHECKPOINT_DIR)
ckpt.mkdir(parents=True, exist_ok=True)

h = hashlib.sha1()
h.update(f"{MODEL_ID}|{MAX_SEQ_LENGTH}|{len(texts)}".encode("utf-8"))
for t in texts:
    h.update(t.encode("utf-8"))
    h.update(b"\0")
signature = h.hexdigest()

sig_file = ckpt / "signature.txt"
if sig_file.exists() and sig_file.read_text().strip() != signature:
    print("Checkpoints antigos sao de outros textos/modelo -- apagando.")
    for old in ckpt.glob("chunk_*.npy"):
        old.unlink()
sig_file.write_text(signature)

n_chunks = (len(texts) + CHUNK_SIZE - 1) // CHUNK_SIZE
pending = [i for i in range(n_chunks) if not (ckpt / f"chunk_{i:05d}.npy").exists()]
print(f"{n_chunks} blocos no total, {n_chunks - len(pending)} ja prontos, {len(pending)} a gerar")

t_start = time.time()
for k, i in enumerate(pending, start=1):
    a, b = i * CHUNK_SIZE, min((i + 1) * CHUNK_SIZE, len(texts))
    emb = model.encode(
        texts[a:b],
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,   # normalizado -> cosseno vira produto escalar
        convert_to_tensor=True,
        show_progress_bar=False,
    ).float().cpu().numpy()
    tmp = ckpt / f"chunk_{i:05d}.tmp.npy"
    np.save(tmp, emb)
    os.replace(tmp, ckpt / f"chunk_{i:05d}.npy")   # so conta como pronto se foi gravado inteiro
    elapsed = time.time() - t_start
    eta = elapsed / k * (len(pending) - k)
    print(f"bloco {i + 1}/{n_chunks} ok | {elapsed / 60:.1f} min decorridos | faltam ~{eta / 60:.1f} min")

17 blocos no total, 0 ja prontos, 17 a gerar


## Juntar, verificar e salvar

Antes de salvar, verifica: número de linhas igual ao de cartas, ausência de `NaN`/`inf` (meia precisão pode estourar) e normas próximas de 1. O arquivo de metadados guarda tudo que o notebook de busca precisa para usar **a mesma configuração**.

In [ ]:
parts = [np.load(ckpt / f"chunk_{i:05d}.npy") for i in range(n_chunks)]
embeddings = np.concatenate(parts).astype(np.float32)

assert embeddings.shape[0] == len(texts), f"{embeddings.shape[0]} vetores para {len(texts)} cartas"
assert np.isfinite(embeddings).all(), "NaN/inf nos vetores -- tente bfloat16 ou float32 na celula do modelo"
norms = np.linalg.norm(embeddings, axis=1)
print(f"shape {embeddings.shape} | norma min {norms.min():.4f} max {norms.max():.4f} (esperado ~1.0)")

np.save(EMBEDDINGS_PATH, embeddings)
meta = {
    "ids": ids,
    "names": names,
    "model_id": MODEL_ID,
    "max_seq_length": MAX_SEQ_LENGTH,
    "query_prefix": QUERY_PREFIX,
    "text_field": TEXT_FIELD,
    "n_cards": len(ids),
    "dim": int(embeddings.shape[1]),
    "texts_sha1": signature,
}
with open(EMBEDDINGS_META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False)

print(f"Embeddings salvos em {EMBEDDINGS_PATH} ({embeddings.nbytes / 1e6:.1f} MB)")
print(f"Metadados salvos em {EMBEDDINGS_META_PATH}")

## Sanity check (não é a busca final)

Algumas consultas com cosseno "na mão", usando o **prefixo com instrução** só na consulta. Serve para ver se os vetores fazem sentido antes de levar tudo para o notebook de busca.

In [ ]:
def encode_query(q):
    v = model.encode([QUERY_PREFIX + q], normalize_embeddings=True, convert_to_tensor=True)
    return v.float().cpu().numpy()[0]


for query in ["creature that destroys artifacts", "creature destroyer artifact", "counter target spell"]:
    scores = embeddings @ encode_query(query)
    print(f"\nConsulta: {query!r}")
    for i in np.argsort(-scores)[:5]:
        print(f"  {scores[i]:.3f}  {names[i]}")

## Próximo passo: usar estes vetores no notebook de busca

No `busca_mtg.ipynb`, troque a configuração pelos valores gravados nos metadados. **O prefixo da consulta precisa ser exatamente o mesmo** usado aqui, e o modelo de busca também precisa de `padding_side="left"`:

```python
import json
from sentence_transformers import SentenceTransformer

meta = json.load(open("Dados/embeddings_qwen3_meta.json", encoding="utf-8"))

EMBEDDINGS_PATH = "Dados/embeddings_qwen3.npy"
MODEL_NAME = meta["model_id"]           # "Qwen/Qwen3-Embedding-0.6B" (ou a pasta local, se estiver offline)
QUERY_PREFIX = meta["query_prefix"]     # "Instruct: ...\nQuery:"

model = SentenceTransformer(MODEL_NAME, tokenizer_kwargs={"padding_side": "left"})
model.max_seq_length = meta["max_seq_length"]
```

Na busca, o modelo roda em CPU com precisão total, enquanto os vetores das cartas foram gerados em meia precisão na GPU; a diferença numérica é pequena. O `assert` que compara o número de cartas com o de vetores continua valendo: ele acusa se o `card_documents.jsonl` mudou desde a geração.